In [75]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA

In [76]:
df = pd.read_excel("processing_data_2.xlsx")

In [77]:
# ===== CHỌN FEATURE =====
drop_cols = [
    "id", "name", "image_path_x", "thumbnail_url", "dup_representative",
    "cluster_id"
]
# chỉ lấy cột còn lại
X = df[[c for c in df.columns if c not in drop_cols]].copy()

# Ép kiểu số cho các cột object nếu có thể (tránh NaN do text số)
for c in X.columns:
    if X[c].dtype == "object":
        conv = pd.to_numeric(X[c], errors="coerce")
        # nếu > 60% giá trị ép được số thì giữ bản ép
        if np.isfinite(conv).mean() > 0.6:
            X[c] = conv

# Lấy phần numeric để đưa qua pipeline
X_num = X.select_dtypes(include=[np.number]).copy()


In [78]:
# ===== IMPUTE + SCALE =====
imputer = SimpleImputer(strategy="median")
X_imp = imputer.fit_transform(X_num)  # xử lý NaN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

In [79]:
# ===== FEATURE SELECTION =====
# Loại bỏ feature gần như hằng số (đặt ngưỡng thấp để chỉ bỏ cột thật sự "constant")
sel = VarianceThreshold(threshold=1e-6)
X_selected = sel.fit_transform(X_scaled)
selected_features = X_num.columns[sel.get_support()]

print("Số feature numeric trước:", X_num.shape[1])
print("Số feature sau khi selection:", len(selected_features))

Số feature numeric trước: 55
Số feature sau khi selection: 49


In [80]:
# ===== DIMENSIONALITY REDUCTION (PCA) =====
# Giữ 95% phương sai
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_selected)

print("Số chiều sau PCA:", X_pca.shape[1])
print("Tỷ lệ phương sai giải thích tích luỹ (các thành phần đầu):",
    np.round(np.cumsum(pca.explained_variance_ratio_)[:10], 4))


Số chiều sau PCA: 23
Tỷ lệ phương sai giải thích tích luỹ (các thành phần đầu): [0.1848 0.2953 0.3962 0.4802 0.5345 0.5845 0.6254 0.6609 0.6957 0.7278]


In [81]:
df = df.copy()
for i in range(X_pca.shape[1]):
    df[f"pca{i+1}"] = X_pca[:, i]

In [82]:
df.to_excel("processing_data_3.xlsx", index=False)
print("[INFO] Đã lưu processing_data_3.xlsx")

[INFO] Đã lưu processing_data_3.xlsx
